# TRAIN RAG SYSTEM
## Загрузка либ

In [1]:
%%capture
!pip install mlcroissant
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers transformers accelerate
!pip install -q -U bitsandbytes
!pip install -q ragas

## Импорты

In [238]:
import logging
import os
import pprint
import time
import warnings

import pandas as pd
import torch
from datasets import Dataset as HFDataset
from mlcroissant import Dataset
from transformers import AutoTokenizer, BitsAndBytesConfig, pipeline

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import AnswerRelevancy, ContextPrecision, Faithfulness
from ragas.run_config import RunConfig

# Загрузка данных

In [3]:
logging.getLogger().setLevel(logging.ERROR)

def clean_text(text):
    if isinstance(text, (bytes, bytearray)):
        return text.decode('utf-8')
    return str(text) if text is not None else ""

ds = Dataset(jsonld="https://huggingface.co/api/datasets/neulab/conala/croissant")
records = ds.records("curated")

incidents = []

for i, record in enumerate(records):
    if i >= 700:
        break

    raw_intent = record.get('curated/intent', "")
    raw_snippet = record.get('curated/snippet', "")
    raw_id = record.get('curated/question_id', i)

    intent = clean_text(raw_intent)
    snippet = clean_text(raw_snippet)

    if not intent or not snippet:
        continue

    incidents.append({
        "incident_id": f"INC-{clean_text(raw_id)}",
        "summary": intent,
        "description": intent,
        "resolution": snippet
    })
    
if incidents:
    print("\nПример готовой записи для FAISS:")
    pprint.pprint(incidents[0])


Пример готовой записи для FAISS:
{'description': 'How can I send a signal from a python program?',
 'incident_id': 'INC-15080500',
 'resolution': 'os.kill(os.getpid(), signal.SIGUSR1)',
 'summary': 'How can I send a signal from a python program?'}


## Создание эмбедингов и добавление их в вектороную DB

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True}
)

documents = [
    Document(
        page_content=item['summary'],
        metadata={
            "incident_id": item['incident_id'],
            "resolution": item['resolution']
        }
    ) for i, item in enumerate(incidents)
]

vector_db = FAISS.from_documents(documents, embeddings)
# vector_db.save_local("faiss_index_support")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [5]:
vector_db.index.ntotal

700

## Тест поиска по FAISS, сравнение `similarity_search` и `similarity_search_with_relevance_scores`

In [33]:
pp = pprint.PrettyPrinter(indent=2)

test_queries = [
    "How to flatten a list of lists in Python?",
    "How to sort a list of dictionaries by a specific key?",
    "Как мне прочитать JSON файл и превратить его в словарь?",
    "How to create a custom callback in PyTorch Lightning to log GPU memory usage?"
]

for i, q in enumerate(test_queries, 1):
    search_result = vector_db.similarity_search(q, k=3)
    print(f"ТЕСТ №{i}: {q}")
    print(f"{' НАЙДЕННЫЕ ДОКУМЕНТЫ ':-^100}")
    
    if not search_result:
        print("Ничего не найдено.")
    else:
        for idx, doc in enumerate(search_result, 1):
            print(f"Документ #{idx} (Score/Metadata: {doc.metadata.get('resolution', 'N/A')})")
            print(f"Содержимое:\n{doc.page_content}")
            print("-" * 30)
    print('\n')

ТЕСТ №1: How to flatten a list of lists in Python?
--------------------------------------- НАЙДЕННЫЕ ДОКУМЕНТЫ ----------------------------------------
Документ #1 (Score/Metadata: [y for x in data for y in (x if isinstance(x, list) else [x])])
Содержимое:
How to flatten a hetrogenous list of list into a single list in python?
------------------------------
Документ #2 (Score/Metadata: [image for menuitem in list_of_menuitems for image in menuitem])
Содержимое:
Flattening a shallow list in Python
------------------------------
Документ #3 (Score/Metadata: df.columns = df.columns.get_level_values(0))
Содержимое:
Python Pandas - How to flatten a hierarchical index in columns
------------------------------


ТЕСТ №2: How to sort a list of dictionaries by a specific key?
--------------------------------------- НАЙДЕННЫЕ ДОКУМЕНТЫ ----------------------------------------
Документ #1 (Score/Metadata: sorted(o.items()))
Содержимое:
How to sort dictionaries by keys in Python
------------------

In [34]:
THRESHOLD = 0.5

for i, q in enumerate(test_queries, 1):
    
    search_results_with_scores = vector_db.similarity_search_with_relevance_scores(q, k=3)
    
    print(f"ТЕСТ №{i}: {q}")
    print(f"{' НАЙДЕННЫЕ ДОКУМЕНТЫ ':-^100}")
    
    if not search_results_with_scores:
        print("База данных не вернула ни одного результата.")
    else:
        for idx, (doc, score) in enumerate(search_results_with_scores, 1):
           
            status = "Выше порога" if score >= THRESHOLD else "Ниже порога"
            
            print(f"Документ #{idx} | Score: {score:.4f} | {status}")
            print(f"Решение: {doc.metadata.get('resolution', 'N/A')}")
            print(f"Содержимое: {doc.page_content}")
            print("-" * 50)

    
    relevant_docs = [d for d, s in search_results_with_scores if s >= THRESHOLD]
    if not relevant_docs:
        print(f"Информации в базе не найдено")
    else:
        pass
    print('\n')

ТЕСТ №1: How to flatten a list of lists in Python?
--------------------------------------- НАЙДЕННЫЕ ДОКУМЕНТЫ ----------------------------------------
Документ #1 | Score: 0.8337 | Выше порога
Решение: [y for x in data for y in (x if isinstance(x, list) else [x])]
Содержимое: How to flatten a hetrogenous list of list into a single list in python?
--------------------------------------------------
Документ #2 | Score: 0.8151 | Выше порога
Решение: [image for menuitem in list_of_menuitems for image in menuitem]
Содержимое: Flattening a shallow list in Python
--------------------------------------------------
Документ #3 | Score: 0.6702 | Выше порога
Решение: df.columns = df.columns.get_level_values(0)
Содержимое: Python Pandas - How to flatten a hierarchical index in columns
--------------------------------------------------


ТЕСТ №2: How to sort a list of dictionaries by a specific key?
--------------------------------------- НАЙДЕННЫЕ ДОКУМЕНТЫ ---------------------------------------

## Загрузка LLM, настройка квантования и параметров генерации

In [20]:
warnings.filterwarnings("ignore")

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

base_pipe = pipeline(
    "text-generation",
    model=model_id,
    tokenizer=tokenizer,
    device_map="auto", 
    model_kwargs={
        "quantization_config": bnb_config,
        "low_cpu_mem_usage": True,
    },
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=base_pipe)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [211]:
def set_llm_params(max_tokens=1500, temp=0.11, rep_penalty=1.05):
    
    base_pipe.model.generation_config.max_new_tokens = max_tokens
    base_pipe.model.generation_config.temperature = temp
    base_pipe.model.generation_config.repetition_penalty = rep_penalty
    base_pipe.model.generation_config.do_sample = True if temp > 0 else False
    base_pipe.model.generation_config.max_length = 7000
    print(f"Настройки применены: tokens={max_tokens}, temp={temp}, rep_penalty={rep_penalty}")

set_llm_params()

Настройки применены: Tokens=1500, Temp=0.11


# Настройка промта для RAG. Описание логики работы RAG системы.

In [217]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Ты — технический эксперт. Пиши ответ СТРОГО по шаблону ниже.\n\n"
        "1. ТРИГГЕР: Если контекст содержит 'NOT_FOUND', начни ответ с: 'Ответ от LLM. ОТЧЕТ ПО ИНЦИДЕНТУ:'. В остальных случаях начни с: 'ОТЧЕТ ПО ИНЦИДЕНТУ:'.\n"
        "2. ПОРЯДОК: Сначала БЛОК КОДА начни с 'Блок кода на python', затем РОВНО ОДИН АБЗАЦ (МАКСИМУМ ДВА ПРЕДЛОЖЕНИЯ) пояснения на русском языке, начни с 'Пояснение'.\n"
        "3. ЗАПРЕТ: Не пиши НИЧЕГО до кода. После второго предложения в блоке 'Пояснение' ПРЕКРАЩАЙ ГЕНЕРАЦИЮ и ничего не пиши дальше."
    )),
    ("user", "КОНТЕКСТ:\n{context}\n\nЗАДАЧА: {question}")
])

In [218]:
def run_support_system(user_query, threshold=0.45):
    docs_with_scores = vector_db.similarity_search_with_relevance_scores(user_query, k=3)

    relevant_docs = [doc for doc, score in docs_with_scores if score >= threshold]

    print(f"Найдено документов выше порога: {len(relevant_docs)}")

    if not relevant_docs:
        context_text = "NOT_FOUND" 
    else:
        context_text = "\n".join([d.metadata.get('resolution', '') for d in relevant_docs])

    raw_answer = (rag_prompt | llm | StrOutputParser()).invoke({
        "context": context_text,
        "question": user_query
    })

    target_header = "ОТЧЕТ ПО ИНЦИДЕНТУ"
    if target_header in raw_answer:
        clean_answer = raw_answer[raw_answer.find(target_header):]
    else:
        clean_answer = raw_answer

    for stop_word in ["Assistant:", "Human:", "<|im_end|>"]:
        clean_answer = clean_answer.split(stop_word)[0]
    
    return {
        "answer": clean_answer.strip()
    }

In [219]:
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [220]:
llm.pipeline.model.generation_config

GenerationConfig {
  "bos_token_id": 1,
  "do_sample": true,
  "eos_token_id": 2,
  "max_length": 7000,
  "max_new_tokens": 1500,
  "repetition_penalty": 1.05,
  "temperature": 0.11
}

## Тест RAG системы в ручном режиме

In [216]:
warnings.filterwarnings("ignore")

for i, q in enumerate(test_queries, 1):
    start_time = time.time()
    print(f"ТЕСТ №{i}: {q}")

    result = run_support_system(q)

    elapsed = time.time() - start_time
    print(f"Время обработки: {elapsed:.2f} сек.")
    print(f"Ответ:\n{result['answer']}")
    print("-" * 50 + "\n")

НАЧАЛО КОМПЛЕКСНОГО ТЕСТИРОВАНИЯ

ТЕСТ №1: How to flatten a list of lists in Python?
DEBUG: Найдено документов выше порога: 3
Время обработки: 25.91 сек.
Ответ:
ОТЧЕТ ПО ИНЦИДЕНТУ:

Блок кода на python:
```
def flatten_list(lst):
    return [item for sublist in lst for item in sublist]
```
Пояснение:
Сначала мы создаем функцию `flatten_list`, которая принимает аргумент `lst` и возвращает новый список, состоящий из всех элементов `lst` и всех подсписков. Внутри функции мы используем понимание списков в Python и генератор списков `[item for sublist in lst for item in sublist]`. Обратите внимание на использование двух циклов `for` для прохода по всем элементам `lst` и всем подспискам. Этот подход эффективен для понимания и решения этой задачи, когда списки вложены на несколько уровней.

В и
--------------------------------------------------

ТЕСТ №2: How to sort a list of dictionaries by a specific key?
DEBUG: Найдено документов выше порога: 3
Время обработки: 24.56 сек.
Ответ:
ОТЧЕТ ПО И

## Тест системы при помощи RAGAS

In [234]:
def evaluate_rag(query, result_dict):
    ragas_llm = LangchainLLMWrapper(llm)
    ragas_emb = LangchainEmbeddingsWrapper(embeddings)

    metrics = [
        Faithfulness(llm=ragas_llm),
        AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb),
        ContextPrecision(llm=ragas_llm)
    ]

    docs = vector_db.similarity_search(query, k=3)
    context_list = [d.page_content for d in docs]

    data_sample = {
        "question": [query],
        "answer": [result_dict['answer']],
        "contexts": [context_list],
        "ground_truth": ["The user wants to list files in a directory and filter them by extension using Python or shell."]
    }

    dataset = HFDataset.from_dict(data_sample)
    
    score = evaluate(
        dataset,
        metrics=metrics,
        run_config=RunConfig(timeout=240, max_retries=3, max_wait=60)
    )
    
    return score.to_pandas()

In [235]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [236]:
query = "How to list all files in a directory and filter them by extension?"

result = run_support_system(query)
config = RunConfig(timeout=240, max_retries=3, max_wait=60)

try:
    eval_results = evaluate_rag(query, result)
    print("\nМЕТРИКИ КАЧЕСТВА УСПЕШНО ПОЛУЧЕНЫ:")
    display(eval_results)
except Exception as e:
    print(f"Ошибка при оценке: {e}")

DEBUG: Найдено документов выше порога: 3


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()



МЕТРИКИ КАЧЕСТВА УСПЕШНО ПОЛУЧЕНЫ:


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision
0,How to list all files in a directory and filter them by extension?,"[Python: How can I find all files with a particular extension?, Find all files in directory with extension .txt, Find all files in directory with extension .txt]","ОТЧЕТ ПО ИНЦИДЕНТУ:\n\nБлок кода на python:\n```\nresults += [each for each in os.listdir(folder) if each.endswith('.c')]\n```\n\nПояснение:\nВ предложенном примере мы используем встроенную функцию `os.listdir()` для возвращения списка всех файлов и папок в указанной папке. Далее, мы фильтруем этот список с помощью оператора вхождения `.endswith()`, который возвращает `True` в случае, если строка заканчивается на указанную подстроку (в данном случае `.c`).\n\nНам нужно будет заменить `folder` на путь к нашей папке и заменить `.c` на расширение, которое мы ищем. В приведенном примере мы используем",The user wants to list files in a directory and filter them by extension using Python or shell.,NaN,0.535996,1.0
